# Community Notes 陣営反応分析

上から順にセルを実行してください。
「★ 設定」セルのパラメータを変えることで、実行時間やトピックを調整できます。

---
## ★ 設定（ここを編集して実行を調整）

In [ ]:
# ====================================================
# ★ 設定: ここを変えて実行時間・対象を調整 ★
# ====================================================

# --- Drive のパス ---
DRIVE_DATA = '/content/drive/Shareddrives/基礎プロジェクト/data'

# --- 結果の保存先 ---
# Shared Drives は読み取り専用の場合があるため、MyDrive に保存
SAVE_DIR = '/content/drive/MyDrive/toriumi_x3_results'

# --- ratings ファイルの読み込み数 ---
# 全8ファイル → 最も正確だが時間がかかる（推定1〜2時間）
# 2ファイル  → 1/4のデータで高速（推定15〜30分）、大体の傾向がわかる
# 1ファイル  → 最速テスト（推定5〜10分）
RATINGS_FILES = 2       # ← 1〜8 の数字を入れる。全部なら 8

# --- ratings の行数制限 ---
# None  → ファイル全量読み込み（推奨）
# 500000 → 各ファイルの先頭50万行のみ（さらに高速テスト）
NROWS = None            # ← None または数字

# --- 分析対象トピック ---
# キーワードを自由に追加・削除・変更できます
# トピックを減らすと実行が速くなります
TOPICS = {
    'vaccine_covid': [
        'vaccine', 'covid', 'pandemic', 'mask', 'booster',
        'pfizer', 'moderna', 'antivax', 'lockdown', 'fauci',
    ],
    'israel_palestine': [
        'israel', 'palestine', 'gaza', 'hamas', 'idf',
        'netanyahu', 'ceasefire', 'zionist', 'hezbollah',
    ],
    'trump': [
        'trump', 'maga', 'indictment', 'mar-a-lago',
        'january 6', 'j6', 'impeach',
    ],
    'immigration': [
        'immigration', 'border', 'migrant', 'asylum',
        'deportation', 'illegal alien', 'refugee', 'caravan',
    ],
    'gun_control': [
        'gun control', 'second amendment', '2nd amendment',
        'shooting', 'nra', 'firearm', 'gun violence',
    ],
    'ALL_POLITICAL': [
        'trump', 'biden', 'democrat', 'republican', 'gop',
        'congress', 'senate', 'election', 'vote', 'ballot',
        'liberal', 'conservative', 'immigration', 'border',
        'abortion', 'gun control', 'vaccine', 'covid',
        'climate change', 'supreme court',
        'israel', 'palestine', 'gaza', 'ukraine', 'russia',
        'woke', 'dei', 'transgender', 'lgbtq', 'censorship',
        'government', 'policy', 'partisan',
    ],
}

# ====================================================
print(f'ratings files: {RATINGS_FILES}/8')
print(f'nrows: {NROWS if NROWS else "全量"}')
print(f'topics: {list(TOPICS.keys())}')
print(f'save dir: {SAVE_DIR}')

---
## 0. セットアップ（以下は変更不要）

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, subprocess, json

if os.path.exists(DRIVE_DATA):
    print('OK: フォルダが見つかりました')
    for d in sorted(os.listdir(DRIVE_DATA)):
        print(f'  {d}')
else:
    print(f'ERROR: {DRIVE_DATA} が見つかりません')
    print()
    sd = '/content/drive/Shareddrives'
    if os.path.exists(sd):
        print('共有ドライブ一覧:')
        for d in os.listdir(sd): print(f'  {d}')
    print('マイドライブ:')
    for d in sorted(os.listdir('/content/drive/MyDrive'))[:20]: print(f'  {d}')

In [ ]:
!git clone https://github.com/hirototoda/toriumi_x3.git /content/toriumi_x3 2>/dev/null || echo 'already cloned'
os.chdir('/content/toriumi_x3')
!git pull
!pip install -q statsmodels

In [ ]:
# data/raw/ にデータを準備（あればスキップ、なければ解凍）
raw_dir = '/content/toriumi_x3/data/raw'
os.makedirs(raw_dir, exist_ok=True)

# --- ratings: 必要な分だけ解凍、余分は削除 ---
ratings_folder = os.path.join(DRIVE_DATA, 'RatingData')
if os.path.exists(ratings_folder):
    all_zips = sorted([f for f in os.listdir(ratings_folder) if f.startswith('ratings-') and f.endswith('.zip')])
    needed = set(z.replace('.zip', '.tsv') for z in all_zips[:RATINGS_FILES])
    # 余分な ratings tsv を削除
    for f in sorted(os.listdir(raw_dir)):
        if f.startswith('ratings-') and f.endswith('.tsv') and f not in needed:
            print(f'  Removing {f} (over limit {RATINGS_FILES})')
            os.remove(os.path.join(raw_dir, f))
    # 必要なファイルがなければ解凍
    for z in all_zips[:RATINGS_FILES]:
        tsv_name = z.replace('.zip', '.tsv')
        if os.path.exists(os.path.join(raw_dir, tsv_name)):
            print(f'  {tsv_name} already exists, skip')
        else:
            print(f'  Unzipping {z} ...')
            subprocess.run(['unzip', '-o', '-q', os.path.join(ratings_folder, z), '-d', raw_dir], check=True)

# --- notes, history: あればスキップ ---
other_targets = [
    ('Notes data', 'notes-'),
    ('Note status history data', 'noteStatusHistory-'),
]
for subfolder, prefix in other_targets:
    folder = os.path.join(DRIVE_DATA, subfolder)
    if not os.path.exists(folder):
        print(f'WARNING: {folder} not found, skip')
        continue
    for f in sorted(os.listdir(folder)):
        if f.startswith(prefix) and f.endswith('.zip'):
            tsv_name = f.replace('.zip', '.tsv')
            if os.path.exists(os.path.join(raw_dir, tsv_name)):
                print(f'  {tsv_name} already exists, skip')
            else:
                print(f'  Unzipping {f} ...')
                subprocess.run(['unzip', '-o', '-q', os.path.join(folder, f), '-d', raw_dir], check=True)

print()
print('=== data/raw/ ===')
for f in sorted(os.listdir(raw_dir)):
    if f.endswith('.tsv'):
        size = os.path.getsize(os.path.join(raw_dir, f)) / (1024**3)
        print(f'  {f}  ({size:.1f} GB)')

---
## 1. 全トピックでパイプライン実行

In [ ]:
nrows_opt = f'--nrows {NROWS}' if NROWS else ''
!python scripts/run_pipeline.py {nrows_opt} --max-rating-files {RATINGS_FILES}

---
## 2. トピック別比較

In [ ]:
with open('/tmp/topics.json', 'w') as f:
    json.dump(TOPICS, f)

nrows_opt = f'--nrows {NROWS}' if NROWS else ''
!python scripts/run_by_topic.py {nrows_opt} --max-rating-files {RATINGS_FILES} --topics-json /tmp/topics.json

---
## 3. 結果の確認

In [ ]:
import pandas as pd

df = pd.read_csv('data/processed/topic_comparison.csv')
display(df)

In [ ]:
targets = pd.read_csv('data/processed/target_notes.csv')
print(f'ターゲットノート数: {len(targets)}')
display(targets.head(20))

In [ ]:
bursts = pd.read_csv('data/processed/bursts.csv')
print(f'バースト数: {len(bursts)}')
print(bursts['burst_type'].value_counts())

---
## 4. 結果を Drive に保存

In [ ]:
os.makedirs(SAVE_DIR, exist_ok=True)
for f in os.listdir('data/processed'):
    src = os.path.join('data/processed', f)
    if f.endswith('.csv') or f.endswith('.png'):
        subprocess.run(['cp', src, SAVE_DIR])
print(f'Done! Results saved to: {SAVE_DIR}')